In [ ]:
!pip install adapters -q

In [ ]:
import pandas as pd
from transformers import TrainingArguments, EarlyStoppingCallback, AutoTokenizer, set_seed
from datasets import Dataset
import joblib
import numpy as np
from google.colab import drive
import os
import json
import zipfile
from sklearn.metrics import f1_score, classification_report
import time
import torch
from transformers.trainer_utils import get_last_checkpoint
import adapters
from adapters import AutoAdapterModel, SeqBnConfig, AdapterTrainer
#SeqBnConfig = Pfeiffer

In [ ]:
drive.mount('/content/drive')
output_dir = "/content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Pfeiffer"
os.makedirs(output_dir, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading the dataset

In [ ]:
with zipfile.ZipFile("aapd.zip") as z:
    with z.open("aapd.json") as f:
        aapd = json.load(f)

In [ ]:
aapd_df_train = pd.DataFrame(aapd["data"]["train"])
aapd_df_val = pd.DataFrame(aapd["data"]["val"])
aapd_df_test = pd.DataFrame(aapd["data"]["test"])

In [ ]:
mlb = joblib.load("mlb.joblib")

In [ ]:
#reusing the  aapd's mlb
aapd_y_train = mlb.transform(aapd_df_train["labels"])
aapd_y_val   = mlb.transform(aapd_df_val["labels"])
aapd_y_test  = mlb.transform(aapd_df_test["labels"])

In [ ]:
aapd_y_train.shape, aapd_y_val.shape, aapd_y_test.shape #ok

((53840, 54), (1000, 54), (1000, 54))

In [ ]:
aapd_X_train = aapd_df_train["text"]
aapd_X_val   = aapd_df_val["text"]
aapd_X_test  = aapd_df_test["text"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# for DistilBER max token length is 512 - the longest abstract has 522 words, so truncation will happen
def tokenize(texts):
    return tokenizer(texts.tolist(), padding="max_length", truncation=True, max_length=512)

train_enc = tokenize(aapd_X_train)
dev_enc   = tokenize(aapd_X_val)
test_enc  = tokenize(aapd_X_test)

In [ ]:
y_train_bin = aapd_y_train.astype(np.float32)
y_dev_bin   = aapd_y_val.astype(np.float32)
y_test_bin  = aapd_y_test.astype(np.float32)

print(mlb.classes_)
print(y_train_bin.shape)

['Adaptation and Self-Organizing Systems' 'Applications'
 'Artificial Intelligence' 'Combinatorics' 'Computation and Language'
 'Computational Complexity'
 'Computational Engineering, Finance, and Science'
 'Computational Geometry' 'Computational Linguistics'
 'Computer Science and Game Theory'
 'Computer Vision and Pattern Recognition' 'Computers and Society'
 'Cryptography and Security' 'Data Analysis, Statistics and Probability'
 'Data Structures and Algorithms' 'Databases' 'Digital Libraries'
 'Discrete Mathematics' 'Disordered Systems and Neural Networks'
 'Distributed, Parallel, and Cluster Computing'
 'Formal Languages and Automata Theory' 'Human-Computer Interaction'
 'Information Retrieval' 'Information Theory (Computer Science)'
 'Information Theory (Mathematics)' 'Logic' 'Logic in Computer Science'
 'Machine Learning (Computer Science)' 'Machine Learning (Statistics)'
 'Mathematical Software' 'Methodology' 'Multiagent Systems' 'Multimedia'
 'Networking and Internet Architect

In [ ]:
id2label = {i: label for i, label in enumerate(mlb.classes_)}
label2id = {label: i for i, label in enumerate(mlb.classes_)}

In [ ]:
train_dataset = Dataset.from_dict({
    "input_ids": train_enc["input_ids"],
    "attention_mask": train_enc["attention_mask"],
    "labels": y_train_bin.astype("float32")})

eval_dataset = Dataset.from_dict({
    "input_ids": dev_enc["input_ids"],
    "attention_mask": dev_enc["attention_mask"],
    "labels": y_dev_bin.astype("float32")})

test_dataset = Dataset.from_dict({
    "input_ids": test_enc["input_ids"],
    "attention_mask": test_enc["attention_mask"],
    "labels": y_test_bin.astype("float32")})

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))   # sigmoid
    preds = (probs >= 0.5).astype(int) #default for now

    labels = labels.astype(int)

    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)

    return {"f1_micro": f1_micro, "f1_macro": f1_macro}

In [ ]:
print("Train labels:", y_train_bin.shape)
print("Val labels:", y_dev_bin.shape)
print("Test labels:", y_test_bin.shape)
print("Number of labels:", len(mlb.classes_))

Train labels: (53840, 54)
Val labels: (1000, 54)
Test labels: (1000, 54)
Number of labels: 54


**Training function**

In [ ]:
def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def reset_cuda_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

#measure vram only for final best config
def get_peak_vram_gb():
    if not torch.cuda.is_available():
        return None

    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / (1024 ** 3)


def run_training(config, seed=0, evaluate_test=False, measure_vram=False, save_report=False):
    set_seed(seed)

    if measure_vram:
        reset_cuda_peak_memory()

    model = AutoAdapterModel.from_pretrained(config["base_model"])
    #remove unnecessary default/pretraining head as a check revealed it is present
    if "default" in model.heads:
        model.delete_head("default")

    #the adapters' library developers wrote on github that multilabel classification head is supported out of the box and can be implemented like this
    model.add_classification_head("aapd", num_labels=len(mlb.classes_),  multilabel=True, id2label=id2label)

    adapter_config = SeqBnConfig(reduction_factor=config["reduction_factor"])
    model.add_adapter("aapd", config=adapter_config, set_active=True)
    model.train_adapter("aapd")

    #check whether the adapters are working
    print("Active adapters:", model.active_adapters)
    print(model.adapter_summary())
    #check the classification head
    print("Trainable classification/head parameters:")
    for name, param in model.named_parameters():
        if param.requires_grad and ("head" in name.lower() or "classifier" in name.lower() or "classification" in name.lower()):
           print(name, param.numel())

    #for the final statistics
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    training_args = TrainingArguments(
        output_dir=config["output_dir"],

        learning_rate=config["learning_rate"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"],

        num_train_epochs=config["num_train_epochs"],
        weight_decay=config["weight_decay"],
        warmup_ratio=config["warmup_ratio"],

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,

        save_total_limit=2,
        fp16=True,
        report_to="none")
    #adapter trainer, as recommended in the library's documentation
    trainer = AdapterTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stopping_patience"])])

    #check
    print("Active adapters after trainer creation:", trainer.model.active_adapters)

    #for resuming if something goes wrong//collab's runtime gets disconnected
    last_checkpoint = None
    if os.path.isdir(config["output_dir"]):
        last_checkpoint = get_last_checkpoint(config["output_dir"])

    if last_checkpoint is not None:
        print(f"Resuming from checkpoint: {last_checkpoint}")
    else:
        print("Starting training from scratch.")


    sync_cuda()
    train_start = time.perf_counter()
    #includes training + epoch validation + checkpoint saving + early stopping + loading best model
    trainer.train(resume_from_checkpoint=last_checkpoint)

    sync_cuda()
    train_time_sec = time.perf_counter() - train_start
    if measure_vram:
        training_peak_vram_gb = get_peak_vram_gb()
    else:
        training_peak_vram_gb = None

    sync_cuda()
    val_start = time.perf_counter()

    val_results = trainer.evaluate(eval_dataset, metric_key_prefix="val")

    sync_cuda()
    val_eval_time_sec = time.perf_counter() - val_start

    result = {
        "model": config["base_model"],
        "dataset": "AAPD",
        "method": "pfeiffer",
        "seed": seed,

        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "num_train_epochs": config["num_train_epochs"],

        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
        "actual_epochs_trained": trainer.state.epoch,


        "train_time_sec": train_time_sec,
        "val_eval_time_sec": val_eval_time_sec,

        "training_peak_vram_gb": training_peak_vram_gb,

        "trainable_params": trainable_params,
        "total_params": total_params,

        "val_f1_macro": val_results["val_f1_macro"],
        "val_f1_micro": val_results["val_f1_micro"]}

    if evaluate_test:
        sync_cuda()
        test_start = time.perf_counter()

        #single forward pass, gives metrics + raw predictions
        test_pred_output = trainer.predict(test_dataset)

        sync_cuda()
        test_eval_time_sec = time.perf_counter() - test_start

        #derive predictions, needed for classification report
        test_probs = 1 / (1 + np.exp(-test_pred_output.predictions))
        test_binary_preds = (test_probs >= 0.5).astype(int)
        gold_labels = (test_pred_output.label_ids >= 0.5).astype(int)

        test_metrics = test_pred_output.metrics

        result.update({
            "test_eval_time_sec": test_eval_time_sec,
            "test_inference_per_sample_ms": (test_eval_time_sec / len(test_dataset)) * 1000,
            "test_f1_macro": test_metrics["test_f1_macro"],
            "test_f1_micro": test_metrics["test_f1_micro"],
            "avg_predicted_labels": float(test_binary_preds.sum(axis=1).mean()),
            "avg_gold_labels": float(gold_labels.sum(axis=1).mean()),})

        if save_report:
            report_dict = classification_report(
                gold_labels, test_binary_preds,
                target_names=mlb.classes_, zero_division=0, output_dict=True)
            report_df = pd.DataFrame(report_dict).T
            report_path = os.path.join(output_dir, f"classification_report_seed_{seed}.csv")
            report_df.to_csv(report_path)
            print(f"Classification report saved to {report_path}")

            #saving raw arrays for possible future analysis
            predictions_path = os.path.join(output_dir, f"test_predictions_seed_{seed}.npz")
            np.savez_compressed(
                predictions_path,
                y_true=gold_labels,
                y_pred=test_binary_preds,
                y_prob=test_probs,
                label_names=np.array(mlb.classes_),
                threshold=np.array([0.5]))
            result["test_predictions_path"] = predictions_path
            print(f"Predictions saved to {predictions_path}")

            #for saving the adapter
            adapter_save_path = os.path.join(config["output_dir"], "final_Pfeiffer_adapter_with_head")
            trainer.model.save_adapter(adapter_save_path, "aapd", with_head=True)
            result["saved_adapter_path"] = adapter_save_path
            print(f"Adapter + head saved to {adapter_save_path}")


    result["total_measured_time_sec"] = (result["train_time_sec"] + result["val_eval_time_sec"] + result.get("test_eval_time_sec", 0))

    return result

**Hyperparameter search**

In [ ]:
#fixed params
base_config = {
    "output_dir": os.path.join(output_dir, "search"),
    "base_model": "distilbert-base-uncased",
    "tokenizer_name": "distilbert-base-uncased",

    "max_length": 512,
    "num_train_epochs": 10, #should be enough for such a large dataset
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "early_stopping_patience": 3,

    "reduction_factor": 8}  #the default one is 16, but since DistilBERT is already a smaller model I will go with 8, this is also the choice of  Razuvayevskaya et al. (2024)

#small search on the most relevant hyperparameters
learning_rates = [1e-4, 2e-4, 5e-4] #1e-4 is recommmended in the adapters library documentation, 2e-4 is used by Razuvayevskaya et al. (2024)
batch_sizes = [8, 16]


search_results_path = os.path.join(output_dir, "search_results.csv")
search_results = []

for lr in learning_rates:
    for bs in batch_sizes:
        config = base_config.copy()
        config["learning_rate"] = lr
        config["batch_size"] = bs
        config["output_dir"] = (f"{base_config['output_dir']}/lr_{lr}_bs_{bs}")

        #skipping already-completed configs on resume
        if os.path.exists(search_results_path):
            existing = pd.read_csv(search_results_path)
            already_done = existing[
                (existing["learning_rate"] == lr) &
                (existing["batch_size"] == bs)]
            if len(already_done) > 0:
                print(f"Skipping lr={lr}, bs={bs} (already done)")
                search_results.append(already_done.iloc[0].to_dict())
                continue

        print("=" * 80)
        print(f"Running DistilBERT: lr={lr}, batch_size={bs}")
        print("=" * 80)

        result = run_training(config, seed=0)
        search_results.append(result)

        #saving incrementally after every config
        pd.DataFrame(search_results).to_csv(search_results_path, index=False)

search_results_df = pd.DataFrame(search_results)
search_results_df = search_results_df.sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
search_results_df.to_csv(search_results_path, index=False)
#!!! train_time_sec in search results is unreliable due to checkpoint resumption !!!
# !!!timing is only reported from the final seed runs - I ensured the run is not resumed

#the warning about adapters can be ignored, based on the check the adapters work
#the warining about early stopping can also be ignored, it works

search_results_df

Skipping lr=0.0001, bs=8 (already done)
Skipping lr=0.0001, bs=16 (already done)
Skipping lr=0.0002, bs=8 (already done)
Skipping lr=0.0002, bs=16 (already done)
Running DistilBERT: lr=0.0005, batch_size=8


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Active adapters: Stack[aapd]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
aapd                     bottleneck          889,920       1.341       1       1
--------------------------------------------------------------------------------
Full model                                66,362,880     100.000               0
Trainable classification/head parameters:
heads.aapd.1.weight 589824
heads.aapd.1.bias 768
heads.aapd.4.weight 41472
heads.aapd.4.bias 54
Active adapters after trainer creation: Stack[aapd]
Resuming from checkpoint: /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Pfeiffer/search/lr_0.0005_bs_8/checkpoint-26920


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
5,0.056400,0.061119,0.737912,0.566160
6,0.052300,0.060692,0.741737,0.572148
7,0.048500,0.061242,0.746039,0.576458
8,0.044900,0.062328,0.745953,0.560451
9,0.041500,0.063737,0.743772,0.582051
10,0.038700,0.064260,0.745037,0.581014


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Running DistilBERT: lr=0.0005, batch_size=16


Active adapters: Stack[aapd]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
aapd                     bottleneck          889,920       1.341       1       1
--------------------------------------------------------------------------------
Full model                                66,362,880     100.000               0
Trainable classification/head parameters:
heads.aapd.1.weight 589824
heads.aapd.1.bias 768
heads.aapd.4.weight 41472
heads.aapd.4.bias 54
Active adapters after trainer creation: Stack[aapd]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.116600,0.075154,0.655612,0.382816
2,0.070300,0.064906,0.720984,0.519109
3,0.064200,0.062229,0.735956,0.537549
4,0.059600,0.059883,0.744046,0.551477
5,0.055500,0.059657,0.743243,0.567038
6,0.051700,0.059652,0.749038,0.585419
7,0.048100,0.060260,0.750675,0.574070
8,0.044700,0.060837,0.750951,0.587413
9,0.041700,0.061415,0.749273,0.581305
10,0.039200,0.061520,0.755329,0.591945


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


,model,dataset,method,seed,learning_rate,batch_size,num_train_epochs,best_checkpoint,best_metric,actual_epochs_trained,train_time_sec,val_eval_time_sec,training_peak_vram_gb,trainable_params,total_params,val_f1_macro,val_f1_micro,total_measured_time_sec
0,distilbert-base-uncased,AAPD,pfeiffer,0,0.0005,16,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.591945,10.0,5679.733455,4.778517,NaN,1522038,67884918,0.591945,0.755329,5684.511972
1,distilbert-base-uncased,AAPD,pfeiffer,0,0.0005,8,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.582051,10.0,3562.575038,5.291852,NaN,1522038,67884918,0.582051,0.743772,3567.866890
2,distilbert-base-uncased,AAPD,pfeiffer,0,0.0002,16,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.577261,10.0,5799.617584,5.151373,NaN,1522038,67884918,0.577261,0.750847,5804.768957
3,distilbert-base-uncased,AAPD,pfeiffer,0,0.0002,8,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.570519,8.0,4727.215723,5.242617,NaN,1522038,67884918,0.570519,0.748581,4732.458340
4,distilbert-base-uncased,AAPD,pfeiffer,0,0.0001,8,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.555670,10.0,5907.462285,5.094197,NaN,1522038,67884918,0.555670,0.740000,5912.556482
5,distilbert-base-uncased,AAPD,pfeiffer,0,0.0001,16,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.552666,10.0,5827.362294,5.116370,NaN,1522038,67884918,0.552666,0.743802,5832.478664


**Best configuration**

In [ ]:
best_row = search_results_df.iloc[0]

best_lr = float(best_row["learning_rate"])
best_batch_size = int(best_row["batch_size"])

print("Best learning rate:", best_lr)
print("Best batch size:", best_batch_size)
print("Best validation macro-F1:", best_row["val_f1_macro"])
print("Best checkpoint:", best_row["best_checkpoint"])

Best learning rate: 0.0005
Best batch size: 16
Best validation macro-F1: 0.5919448896977693
Best checkpoint: /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Pfeiffer/search/lr_0.0005_bs_16/checkpoint-33650


In [ ]:
best_config = base_config.copy()
best_config["learning_rate"] = best_lr
best_config["batch_size"] = best_batch_size
best_config["selection_metric"] = "val_f1_macro"
best_config["best_validation_macro_f1"] = float(best_row["val_f1_macro"])
best_config["best_validation_micro_f1"] = float(best_row["val_f1_micro"])
best_config["best_checkpoint_from_search"] = best_row["best_checkpoint"]

best_config_path = os.path.join(output_dir, "best_config.json")

with open(best_config_path, "w") as f:
    json.dump(best_config, f, indent=2)


**Final run (test set) on the best found configuration**

In [ ]:
with open(best_config_path, "r") as f:
    final_config = json.load(f)

In [ ]:
test_output_dir = os.path.join(output_dir, "test")
os.makedirs(test_output_dir, exist_ok=True)

test_results_path = os.path.join(test_output_dir, "AAPD_DistilBERT_Pfeiffer_test_results.csv")
test_results = []

for seed in [0, 1, 2]:
    config = final_config.copy()
    config["seed"] = seed
    config["output_dir"] = (os.path.join(test_output_dir, f"AAPD_DistilBERT_Pfeiffer_test_seed_{seed}"))

    # skips already-completed seeds on resume
    if os.path.exists(test_results_path):
        existing = pd.read_csv(test_results_path)
        already_done = existing[existing["seed"] == seed]
        if len(already_done) > 0:
            print(f"Skipping seed={seed} (already done)")
            test_results.append(already_done.iloc[0].to_dict())
            continue

    print("=" * 80)
    print(f"Final run: seed={seed}, lr={final_config['learning_rate']}, batch_size={final_config['batch_size']}")
    print("=" * 80)
    result = run_training(config, seed=seed, evaluate_test=True, measure_vram=True, save_report = True)
    test_results.append(result)

    #saves incrementally after every seed
    pd.DataFrame(test_results).to_csv(test_results_path, index=False)

test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv(test_results_path, index=False)
test_results_df

Final run: seed=0, lr=0.0005, batch_size=16


Active adapters: Stack[aapd]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
aapd                     bottleneck          889,920       1.341       1       1
--------------------------------------------------------------------------------
Full model                                66,362,880     100.000               0
Trainable classification/head parameters:
heads.aapd.1.weight 589824
heads.aapd.1.bias 768
heads.aapd.4.weight 41472
heads.aapd.4.bias 54
Active adapters after trainer creation: Stack[aapd]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.116600,0.074538,0.661066,0.391555
2,0.070300,0.065206,0.723109,0.522079
3,0.064200,0.061176,0.743415,0.546218
4,0.059600,0.059422,0.749595,0.553260
5,0.055600,0.059189,0.751529,0.571413
6,0.051800,0.059502,0.749433,0.573675
7,0.048300,0.060092,0.748809,0.568057
8,0.044800,0.060812,0.747307,0.573580
9,0.041900,0.061712,0.746595,0.579529
10,0.039400,0.061755,0.747726,0.586161


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Classification report saved to /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Pfeiffer/classification_report_seed_0.csv
Predictions saved to /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Pfeiffer/test_predictions_seed_0.npz
Adapter + head saved to /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Pfeiffer/test/AAPD_DistilBERT_Pfeiffer_test_seed_0/final_Pfeiffer_adapter_with_head
Final run: seed=1, lr=0.0005, batch_size=16


...

Active adapters: Stack[aapd]
Name                     Architecture         #Param      %Param  Active   Train
--------------------------------------------------------------------------------
aapd                     bottleneck          889,920       1.341       1       1
--------------------------------------------------------------------------------
Full model                                66,362,880     100.000               0
Trainable classification/head parameters:
heads.aapd.1.weight 589824
heads.aapd.1.bias 768
heads.aapd.4.weight 41472
heads.aapd.4.bias 54
Active adapters after trainer creation: Stack[aapd]
Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.117800,0.076309,0.653042,0.388538
2,0.070300,0.065019,0.721044,0.516452
3,0.064200,0.062244,0.736528,0.539968
4,0.059600,0.059648,0.740143,0.538415
5,0.055700,0.059829,0.742133,0.559297
6,0.051900,0.059740,0.747971,0.579954
7,0.048200,0.060468,0.746370,0.571460
8,0.044900,0.060818,0.749944,0.583447
9,0.042000,0.061472,0.755405,0.591175
10,0.039500,0.061630,0.757338,0.595506


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Classification report saved to /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Pfeiffer/classification_report_seed_2.csv
Predictions saved to /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Pfeiffer/test_predictions_seed_2.npz
Adapter + head saved to /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Pfeiffer/test/AAPD_DistilBERT_Pfeiffer_test_seed_2/final_Pfeiffer_adapter_with_head


,model,dataset,method,seed,learning_rate,batch_size,num_train_epochs,best_checkpoint,best_metric,actual_epochs_trained,...,val_f1_micro,test_eval_time_sec,test_inference_per_sample_ms,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,test_predictions_path,saved_adapter_path,total_measured_time_sec
0,distilbert-base-uncased,AAPD,pfeiffer,0,0.0005,16,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.586161,10.0,...,0.747726,5.101252,5.101252,0.587281,0.736307,2.107,2.421,/content/drive/MyDrive/thesis_results/AAPD_Dis...,/content/drive/MyDrive/thesis_results/AAPD_Dis...,5658.549252
1,distilbert-base-uncased,AAPD,pfeiffer,1,0.0005,16,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.602990,10.0,...,0.760257,4.973131,4.973131,0.591261,0.739198,2.092,2.421,/content/drive/MyDrive/thesis_results/AAPD_Dis...,/content/drive/MyDrive/thesis_results/AAPD_Dis...,5665.438420
2,distilbert-base-uncased,AAPD,pfeiffer,2,0.0005,16,10,/content/drive/MyDrive/thesis_results/AAPD_Dis...,0.595506,10.0,...,0.757338,4.843099,4.843099,0.556187,0.731370,2.061,2.421,/content/drive/MyDrive/thesis_results/AAPD_Dis...,/content/drive/MyDrive/thesis_results/AAPD_Dis...,5664.982349


In [ ]:
test_summary_df = test_results_df[[
    "test_f1_macro",
    "test_f1_micro",
    "avg_predicted_labels",
    "avg_gold_labels",
    "train_time_sec",
    "val_eval_time_sec",
    "test_eval_time_sec",
    "test_inference_per_sample_ms",
    "training_peak_vram_gb",
    "actual_epochs_trained",
    "trainable_params",
    "total_params",
    "total_measured_time_sec"]].agg(["mean", "std"])

test_summary_path =  os.path.join(test_output_dir, "AAPD_DistilBERT_Pfeiffer_test_results_summary.csv")
test_summary_df.to_csv(test_summary_path)

test_summary_df

,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,train_time_sec,val_eval_time_sec,test_eval_time_sec,test_inference_per_sample_ms,training_peak_vram_gb,actual_epochs_trained,trainable_params,total_params,total_measured_time_sec
mean,0.578243,0.735625,2.086667,2.421,5653.158380,4.859133,4.972494,4.972494,1.497319,10.0,1522038.0,67884918.0,5662.990007
std,0.019204,0.003958,0.023459,0.000,3.962805,0.073813,0.129078,0.129078,0.001895,0.0,0.0,0.0,3.852561
